In [0]:
import datetime as dt
import requests
import json
# ---------------- Config ----------------
CATALOG = "canada_business"
SCHEMA  = "bronze"
BRONZE_SUBDIR = "tariff_raw_html"

URL = "https://en.wikipedia.org/wiki/United_States%E2%80%93China_trade_war"

# ---------------- Helpers (same style as your other ingests) ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")

def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()
    rows = df.loc[df["database_description_item"] == "RootLocation", "database_description_value"].values
    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(f"RootLocation not found for {catalog}.{schema}")
    return str(rows[0])

def join_uri(base_uri: str, child: str) -> str:
    return base_uri.rstrip("/") + "/" + child.strip("/")

# ---------------- Paths ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)

data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"

dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)

# ---------------- Fetch HTML ----------------
ts = run_ts_utc()
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(URL, timeout=60, headers=headers)
r.raise_for_status()
html = r.text

# ---------------- Write raw HTML ----------------
out_file = join_uri(data_base, f"us_china_trade_war_{ts}.html")
dbutils.fs.put(out_file, html, overwrite=False)

manifest = {
    "run_ts": ts,
    "source": "wikipedia",
    "url": URL,
    "raw_file": out_file,
    "bytes": len(html)
}

manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)

print("✅ Tariff HTML ingest complete")
print("Bronze base:", bronze_base)
print("Raw file:", out_file)
print("Manifest:", manifest_path)
print("Bytes:", len(html))